In [10]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import json


class ImageLSH:
    def __init__(self, num_hash_tables):
        self.num_hash_tables = num_hash_tables
        self.hash_tables = [{} for _ in range(num_hash_tables)]
    
    def hash_function(self, image_hash):
        if not image_hash:
            return 0
        return int(str(image_hash), 16) % self.num_hash_tables
    
    def index(self, hash_values):
        for hash_value in hash_values:
            for table_idx in range(self.num_hash_tables):
                bucket = self.hash_function(hash_value)
                if bucket not in self.hash_tables[table_idx]:
                    self.hash_tables[table_idx][bucket] = []
                self.hash_tables[table_idx][bucket].append(hash_value)

    @staticmethod
    def dhash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # dHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity

    @staticmethod
    def ahash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # AHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity
    
    @staticmethod
    def phash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # pHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity

    @staticmethod
    def chash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # cHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity

    def calculate_similarity(self, hash1, hash2, method):
        if method == 'dhash':
            return ImageLSH.dhash_similarity(hash1, hash2)
        elif method == 'ahash':
            return ImageLSH.ahash_similarity(hash1, hash2)
        elif method == 'phash':
            return ImageLSH.phash_similarity(hash1, hash2)
        elif method == 'chash':
            return ImageLSH.chash_similarity(hash1, hash2)
        else:
            raise ValueError("Invalid similarity calculation method")


# Function to extract hash values from the API
def extract_hash(product_url):
    url = "http://192.168.131.170/cp/api/v1/extract-avg-hash"
    payload = {'product_url': product_url}
    
    response = requests.post(url, data=payload)
    
    if response.status_code == 200:
        data = response.json()
        # Extract only the required hashes from the response
        response_data = data.get('data', {}).get('response', {})
        return {
            'ahash': response_data.get('average_hash', ''),
            'chash': response_data.get('color_hash', ''),
            'dhash': response_data.get('difference_hash', ''),
            'phash': response_data.get('perceptual_hash', '')
        }
    else:
        return {
            'ahash': '',
            'chash': '',
            'dhash': '',
            'phash': ''
        }

# Extract and get all hashes using threading
def get_all_hashes(df):
    hash_results = []
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(extract_hash, url) for url in df['image_url']]
        for future in futures:
            hash_results.append(future.result())
    
    # Add the extracted hash values to the DataFrame
    df['hash_data'] = hash_results
    return df


# Filter similar images based on hash similarity
def check_for_duplicates(df, lsh):
    # Initialize placeholder for similar image information
    df['similar_images'] = ''

    # Iterate through each image hash in the DataFrame
    for i, row in df.iterrows():
        query_ahash = row['hash_data'].get('ahash')
        query_dhash = row['hash_data'].get('dhash')
        query_phash = row['hash_data'].get('phash')
        query_chash = row['hash_data'].get('chash')

        similarities = []
        for j, candidate_row in df.iterrows():
            if i == j:
                continue  # Skip comparing with itself

            # Calculate similarities for each type of hash
            ahash_sim = lsh.calculate_similarity(query_ahash, candidate_row['hash_data'].get('ahash'), 'ahash')
            dhash_sim = lsh.calculate_similarity(query_dhash, candidate_row['hash_data'].get('dhash'), 'dhash')
            phash_sim = lsh.calculate_similarity(query_phash, candidate_row['hash_data'].get('phash'), 'phash')
            chash_sim = lsh.calculate_similarity(query_chash, candidate_row['hash_data'].get('chash'), 'chash')

            # Filter and store results based on similarity score thresholds
            if ahash_sim >= 0.65:
                similarities.append({'pid': candidate_row['product_id'], 'score': round(ahash_sim, 3), 'hash_type': 'ahash'})
            if dhash_sim >= 0.65:
                similarities.append({'pid': candidate_row['product_id'], 'score': round(dhash_sim, 3), 'hash_type': 'dhash'})
            if phash_sim >= 0.65:
                similarities.append({'pid': candidate_row['product_id'], 'score': round(phash_sim, 3), 'hash_type': 'phash'})
            if chash_sim >= 0.95:
                similarities.append({'pid': candidate_row['product_id'], 'score': round(chash_sim, 3), 'hash_type': 'chash'})

        # Store the similarity information as JSON
        df.at[i, 'similar_images'] = json.dumps(similarities)

    return df


# Save the final DataFrame to a CSV file
def save_to_csv(df, file_path):
    df.to_csv(file_path, index=False)


def main():
    # Load the CSV file
    file_path = '/home/justdial/Downloads/Need Image Hash Code.csv'  # Update with your actual file path
    df = pd.read_csv(file_path)

    # df = df.head()

    # Create LSH instance
    lsh = ImageLSH(num_hash_tables=10)

    # Extract hash values for each image URL
    df_with_hashes = get_all_hashes(df)

    # Identify duplicates based on hash similarity
    df_with_duplicates = check_for_duplicates(df_with_hashes, lsh)

    print(df_with_duplicates)

    # Save the results back to CSV
    output_file_path = '/home/justdial/Downloads/Need Image Hash Code_with_duplicates_final.csv'
    save_to_csv(df_with_duplicates, output_file_path)

    print(f"Results saved to {output_file_path}")

    return df_with_duplicates

if __name__ == "__main__":
    df = main()


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [9]:
df

,product_id,image_url,hash_data,similar_images
0,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e3f7f7f7f3f7e3e3', 'chash': '07e000...","[{""pid"": 227447977, ""score"": 0.859, ""hash_type..."
1,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e7e7c7e3e3e3e3e3', 'chash': '06c002...","[{""pid"": 227447977, ""score"": 0.859, ""hash_type..."
2,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'ffe7e3e3e7e3e3e7', 'chash': '076006...","[{""pid"": 227447977, ""score"": 0.797, ""hash_type..."
3,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f3f3c38181e3c3c3', 'chash': '07e000...","[{""pid"": 227447977, ""score"": 0.719, ""hash_type..."
4,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'ffff180000000000', 'chash': '194016...","[{""pid"": 227447977, ""score"": 0.656, ""hash_type..."


In [9]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import json


class ImageLSH:
    def __init__(self, num_hash_tables):
        self.num_hash_tables = num_hash_tables
        self.hash_tables = [{} for _ in range(num_hash_tables)]
    
    def hash_function(self, image_hash):
        if not image_hash:
            return 0
        return int(str(image_hash), 16) % self.num_hash_tables
    
    def index(self, hash_values):
        for hash_value in hash_values:
            for table_idx in range(self.num_hash_tables):
                bucket = self.hash_function(hash_value)
                if bucket not in self.hash_tables[table_idx]:
                    self.hash_tables[table_idx][bucket] = []
                self.hash_tables[table_idx][bucket].append(hash_value)

    @staticmethod
    def dhash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # dHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity

    @staticmethod
    def ahash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # AHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity
    
    @staticmethod
    def phash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # pHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity

    @staticmethod
    def chash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # cHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity

    def calculate_similarity(self, hash1, hash2, method):
        if method == 'dhash':
            return ImageLSH.dhash_similarity(hash1, hash2)
        elif method == 'ahash':
            return ImageLSH.ahash_similarity(hash1, hash2)
        elif method == 'phash':
            return ImageLSH.phash_similarity(hash1, hash2)
        elif method == 'chash':
            return ImageLSH.chash_similarity(hash1, hash2)
        else:
            raise ValueError("Invalid similarity calculation method")


# Function to extract hash values from the API
def extract_hash(product_url):
    url = "http://192.168.131.170/cp/api/v1/extract-avg-hash"
    payload = {'product_url': product_url}
    
    response = requests.post(url, data=payload)
    
    if response.status_code == 200:
        data = response.json()
        # Extract only the required hashes from the response
        response_data = data.get('data', {}).get('response', {})
        return {
            'ahash': response_data.get('average_hash', ''),
            'chash': response_data.get('color_hash', ''),
            'dhash': response_data.get('difference_hash', ''),
            'phash': response_data.get('perceptual_hash', '')
        }
    else:
        return {
            'ahash': '',
            'chash': '',
            'dhash': '',
            'phash': ''
        }

# Extract and get all hashes using threading
def get_all_hashes(df):
    hash_results = []
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(extract_hash, url) for url in df['image_url']]
        for future in futures:
            hash_results.append(future.result())
    
    # Add the extracted hash values to the DataFrame
    df['hash_data'] = hash_results
    return df


def check_for_duplicates(df, lsh):
    # Initialize placeholder for similar image information
    df['similar_images'] = ''
    total_rows = len(df)

    # Iterate through each image hash in the DataFrame
    for i, row in df.iterrows():
        query_ahash = row['hash_data'].get('ahash')
        query_dhash = row['hash_data'].get('dhash')
        query_phash = row['hash_data'].get('phash')
        query_chash = row['hash_data'].get('chash')

        similarities = []
        for j, candidate_row in df.iterrows():
            if i == j:
                continue  # Skip comparing with itself

            # Calculate similarities for each type of hash
            ahash_sim = lsh.calculate_similarity(query_ahash, candidate_row['hash_data'].get('ahash'), 'ahash')
            dhash_sim = lsh.calculate_similarity(query_dhash, candidate_row['hash_data'].get('dhash'), 'dhash')
            phash_sim = lsh.calculate_similarity(query_phash, candidate_row['hash_data'].get('phash'), 'phash')
            chash_sim = lsh.calculate_similarity(query_chash, candidate_row['hash_data'].get('chash'), 'chash')

            # Filter and store results based on similarity score thresholds
            if ahash_sim >= 0.80:
                similarities.append({
                    'product_url': candidate_row['image_url'], 
                    'score': round(ahash_sim, 3), 
                    'hash_type': 'ahash'
                })
            if dhash_sim >= 0.89:
                similarities.append({
                    'product_url': candidate_row['image_url'], 
                    'score': round(dhash_sim, 3), 
                    'hash_type': 'dhash'
                })
            if phash_sim >= 0.90:
                similarities.append({
                    'product_url': candidate_row['image_url'], 
                    'score': round(phash_sim, 3), 
                    'hash_type': 'phash'
                })
            if chash_sim >= 0.99:
                similarities.append({
                    'product_url': candidate_row['image_url'], 
                    'score': round(chash_sim, 3), 
                    'hash_type': 'chash'
                })

        # Store the similarity information as JSON
        df.at[i, 'similar_images'] = json.dumps(similarities)

        # Print progress after processing each row
        print(f"Processed row {i+1}/{total_rows}")
    
    print(f"Total rows processed: {total_rows}")
    return df

# Save the final DataFrame to a CSV file
def save_to_csv(df, file_path):
    df.to_csv(file_path, index=False)


def main():
    # Load the CSV file
    file_path = '/home/justdial/Downloads/Need Image Hash Code.csv'  # Update with your actual file path
    df = pd.read_csv(file_path)

    df = df.head(20)  # Process a subset for testing

    # Create LSH instance
    lsh = ImageLSH(num_hash_tables=10)

    # Extract hash values for each image URL
    df_with_hashes = get_all_hashes(df)

    # Identify duplicates based on hash similarity
    df_with_duplicates = check_for_duplicates(df_with_hashes, lsh)

    print(df_with_duplicates)

    # Save the results back to CSV
    output_file_path = '/home/justdial/Downloads/Need Image Hash Code_with_duplicates_final.csv'
    save_to_csv(df_with_duplicates, output_file_path)

    print(f"Results saved to {output_file_path}")

    return df_with_duplicates


if __name__ == "__main__":
    df = main()


Processed row 1/20
Processed row 2/20
Processed row 3/20
Processed row 4/20
Processed row 5/20
Processed row 6/20
Processed row 7/20
Processed row 8/20
Processed row 9/20
Processed row 10/20
Processed row 11/20
Processed row 12/20
Processed row 13/20
Processed row 14/20
Processed row 15/20
Processed row 16/20
Processed row 17/20
Processed row 18/20
Processed row 19/20
Processed row 20/20
Total rows processed: 20
    product_id                                          image_url  \
0    227447977  http://content.jdmagicbox.com/quickquotes/imag...   
1    227447977  http://content.jdmagicbox.com/quickquotes/imag...   
2    227447977  http://content.jdmagicbox.com/quickquotes/imag...   
3    227447977  http://content.jdmagicbox.com/quickquotes/imag...   
4    227447977  http://content.jdmagicbox.com/quickquotes/imag...   
5    227447977  http://content.jdmagicbox.com/quickquotes/imag...   
6    227447977  http://content.jdmagicbox.com/quickquotes/imag...   
7    227447977  http://content.j

In [8]:
df

,product_id,image_url,hash_data,similar_images
0,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e3f7f7f7f3f7e3e3', 'chash': '07e000...","[{""product_url"": ""http://content.jdmagicbox.co..."
1,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e7e7c7e3e3e3e3e3', 'chash': '06c002...","[{""product_url"": ""http://content.jdmagicbox.co..."
2,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'ffe7e3e3e7e3e3e7', 'chash': '076006...","[{""product_url"": ""http://content.jdmagicbox.co..."
3,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f3f3c38181e3c3c3', 'chash': '07e000...","[{""product_url"": ""http://content.jdmagicbox.co..."
4,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'ffff180000000000', 'chash': '194016...","[{""product_url"": ""http://content.jdmagicbox.co..."
5,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f7c3c18181c1c3c3', 'chash': '130001...","[{""product_url"": ""http://content.jdmagicbox.co..."
6,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f7e7c3e3e7e7e7e7', 'chash': '062000...","[{""product_url"": ""http://content.jdmagicbox.co..."
7,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f3f3c3c181c3c383', 'chash': '07e000...","[{""product_url"": ""http://content.jdmagicbox.co..."
8,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'efefc3c3c3c1c3e3', 'chash': '06e000...","[{""product_url"": ""http://content.jdmagicbox.co..."
9,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'dfdf8787838787c7', 'chash': '0f600c...",[]


In [9]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import json
from tqdm import tqdm


class ImageLSH:
    def __init__(self, num_hash_tables):
        self.num_hash_tables = num_hash_tables
        self.hash_tables = [{} for _ in range(num_hash_tables)]
    
    def hash_function(self, image_hash):
        if not image_hash:
            return 0
        return int(str(image_hash), 16) % self.num_hash_tables
    
    def index(self, hash_values):
        for hash_value in hash_values:
            for table_idx in range(self.num_hash_tables):
                bucket = self.hash_function(hash_value)
                if bucket not in self.hash_tables[table_idx]:
                    self.hash_tables[table_idx][bucket] = []
                self.hash_tables[table_idx][bucket].append(hash_value)

    @staticmethod
    def dhash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # dHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity

    @staticmethod
    def ahash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # AHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity
    
    @staticmethod
    def phash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # pHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity

    @staticmethod
    def chash_similarity(hash1, hash2):
        if not hash1 or not hash2:
            return 0.0 
        hamming_distance = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
        max_distance = 64  # cHash is 64 bits
        similarity = 1 - (hamming_distance / max_distance)
        return similarity

    def calculate_similarity(self, hash1, hash2, method):
        if method == 'dhash':
            return ImageLSH.dhash_similarity(hash1, hash2)
        elif method == 'ahash':
            return ImageLSH.ahash_similarity(hash1, hash2)
        elif method == 'phash':
            return ImageLSH.phash_similarity(hash1, hash2)
        elif method == 'chash':
            return ImageLSH.chash_similarity(hash1, hash2)
        else:
            raise ValueError("Invalid similarity calculation method")


def extract_hash(product_url):
    url = "http://192.168.131.170/cp/api/v1/extract-avg-hash"
    payload = {'product_url': product_url}
    
    response = requests.post(url, data=payload)
    
    if response.status_code == 200:
        data = response.json()
        response_data = data.get('data', {}).get('response', {})
        return {
            'ahash': response_data.get('average_hash', ''),
            'chash': response_data.get('color_hash', ''),
            'dhash': response_data.get('difference_hash', ''),
            'phash': response_data.get('perceptual_hash', '')
        }
    else:
        return {
            'ahash': '',
            'chash': '',
            'dhash': '',
            'phash': ''
        }


def get_all_hashes(df):
    hash_results = []
    
    # Use tqdm to create a progress bar
    with ThreadPoolExecutor(max_workers=10) as executor:
        # Create a list of futures
        futures = [executor.submit(extract_hash, url) for url in df['image_url']]
        
        # Use tqdm to track the progress of futures
        for future in tqdm(futures, desc="Processing URLs", total=len(futures)):
            hash_results.append(future.result())
    
    df['hash_data'] = hash_results
    return df


import json

def check_for_duplicates(df, lsh):
    df['similar_images'] = ''
    total_rows = len(df)

    for i, row in df.iterrows():
        query_ahash = row['hash_data'].get('ahash')
        query_dhash = row['hash_data'].get('dhash')
        query_phash = row['hash_data'].get('phash')
        query_chash = row['hash_data'].get('chash')

        similarities = []
        for j, candidate_row in df.iterrows():
            if i == j or row['product_id'] != candidate_row['product_id']:
                continue  # Skip if it's the same row or if product_ids do not match

            ahash_sim = lsh.calculate_similarity(query_ahash, candidate_row['hash_data'].get('ahash'), 'ahash')
            dhash_sim = lsh.calculate_similarity(query_dhash, candidate_row['hash_data'].get('dhash'), 'dhash')
            phash_sim = lsh.calculate_similarity(query_phash, candidate_row['hash_data'].get('phash'), 'phash')
            chash_sim = lsh.calculate_similarity(query_chash, candidate_row['hash_data'].get('chash'), 'chash')

            if ahash_sim >= 0.80:
                similarities.append({
                    'product_url': candidate_row['image_url'],
                    'score': round(ahash_sim, 3),
                    'hash_type': 'ahash'
                })
            if dhash_sim >= 0.89:
                similarities.append({
                    'product_url': candidate_row['image_url'],
                    'score': round(dhash_sim, 3),
                    'hash_type': 'dhash'
                })
            if phash_sim >= 0.90:
                similarities.append({
                    'product_url': candidate_row['image_url'],
                    'score': round(phash_sim, 3),
                    'hash_type': 'phash'
                })
            if chash_sim >= 0.99:
                similarities.append({
                    'product_url': candidate_row['image_url'],
                    'score': round(chash_sim, 3),
                    'hash_type': 'chash'
                })

        if similarities:
            similarities.insert(0, {
                'product_url': row['image_url'],
                'score': 'self',
                'hash_type': 'self'
            })

        df.at[i, 'similar_images'] = json.dumps(similarities)

        print(f"Processed row {i + 1}/{total_rows}")

    print(f"Total rows processed: {total_rows}")
    return df


def check_for_exact_duplicates(df):
    df['exact_duplicates'] = ''
    total_rows = len(df)

    for i, row in df.iterrows():
        # Initialize a list to hold exact duplicates
        exact_duplicates = []

        for j, candidate_row in df.iterrows():
            # Skip if it's the same row or if product_ids do not match
            if i == j or row['product_id'] != candidate_row['product_id']:
                continue

            # Check if all hash values match
            if (row['hash_data'].get('ahash') == candidate_row['hash_data'].get('ahash') and
                row['hash_data'].get('dhash') == candidate_row['hash_data'].get('dhash') and
                row['hash_data'].get('phash') == candidate_row['hash_data'].get('phash') and
                row['hash_data'].get('chash') == candidate_row['hash_data'].get('chash')):
                
                exact_duplicates.append(candidate_row['image_url'])

        if exact_duplicates:
            # Include the current image URL as well
            exact_duplicates.insert(0, row[['product_id','image_url']])
        
        df.at[i, 'exact_duplicates'] = json.dumps(exact_duplicates)

        print(f"Processed row {i + 1}/{total_rows}")

    print(f"Total rows processed: {total_rows}")
    return df



def save_to_csv(df, output_file_path):
    df.to_csv(output_file_path, index=False)
    

def main():
    # Load the CSV file
    file_path = '/home/justdial/Downloads/Need Image Hash Code.csv'  # Update with your actual file path
    df = pd.read_csv(file_path)

    df = df.head(10000)  # Process a subset for testing

    # Create LSH instance
    lsh = ImageLSH(num_hash_tables=10)

    # Extract hash values for each image URL
    df_with_hashes = get_all_hashes(df)

    # Identify duplicates based on hash similarity
    df_with_duplicates = check_for_exact_duplicates(df_with_hashes)

    print(df_with_duplicates)

    # Save the results back to CSV
    output_file_path = '/home/justdial/Downloads/Need Image Hash Code_with_duplicates_full_&_final6.csv'
    save_to_csv(df_with_duplicates, output_file_path)

    print(f"Results saved to {output_file_path}")

    return df_with_duplicates


if __name__ == "__main__":
    df = main()


Processing URLs:   3%|▎         | 270/10000 [06:41<4:01:15,  1.49s/it]


KeyboardInterrupt: 

In [8]:
df

,product_id,image_url,hash_data,exact_duplicates
0,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e3f7f7f7f3f7e3e3', 'chash': '07e000...",[]
1,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e7e7c7e3e3e3e3e3', 'chash': '06c002...",[]
2,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'ffe7e3e3e7e3e3e7', 'chash': '076006...",[]
3,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f3f3c38181e3c3c3', 'chash': '07e000...",[]
4,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'ffff180000000000', 'chash': '194016...",[]
...,...,...,...,...
1019,227448003,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f7f7e1e1e1e1e1f1', 'chash': '0fc030...",[]
1020,227448003,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e7f9908080808080', 'chash': '014000...",[]
1021,227448003,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f3f3c38181e3c3c3', 'chash': '07e000...",[]
1022,227448003,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e7e7c7c7c7c7e3e3', 'chash': '072000...",[]


In [10]:
df

,product_id,image_url,hash_data,similar_images
0,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e3f7f7f7f3f7e3e3', 'chash': '07e000...","[{""product_url"": ""http://content.jdmagicbox.co..."
1,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e7e7c7e3e3e3e3e3', 'chash': '06c002...","[{""product_url"": ""http://content.jdmagicbox.co..."
2,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'ffe7e3e3e7e3e3e7', 'chash': '076006...","[{""product_url"": ""http://content.jdmagicbox.co..."
3,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f3f3c38181e3c3c3', 'chash': '07e000...","[{""product_url"": ""http://content.jdmagicbox.co..."
4,227447977,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'ffff180000000000', 'chash': '194016...","[{""product_url"": ""http://content.jdmagicbox.co..."
...,...,...,...,...
995,227448003,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e7e7e7ffc9c3f3c3', 'chash': '07e000...","[{""product_url"": ""http://content.jdmagicbox.co..."
996,227448003,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f7c3c18181c1c3c3', 'chash': '130001...","[{""product_url"": ""http://content.jdmagicbox.co..."
997,227448003,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'f7c3c18181c1c1e3', 'chash': '030000...","[{""product_url"": ""http://content.jdmagicbox.co..."
998,227448003,http://content.jdmagicbox.com/quickquotes/imag...,"{'ahash': 'e7e7f7e7e7c7e7e7', 'chash': '07c000...","[{""product_url"": ""http://content.jdmagicbox.co..."
